In [1]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import numpy as np
import pandas as pd
import polars as pl

import ast
import glob
import pickle
import dask
import os
import itertools

from tqdm.notebook import tqdm

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score


from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed

from concurrent.futures import ThreadPoolExecutor

from pprint import pprint
import os

### Load Augmented DF to get the ground truth

In [2]:
augmented_df = pl.read_parquet("../benchmark_LLF/imputed_augmented_us-counties-states_latest_variants.parquet").to_pandas()
augmented_df["date"] = pd.to_datetime(augmented_df["date"])
augmented_df["fips"] = augmented_df["fips"].astype(int)
augmented_df["days_from_start"] = augmented_df["days_from_start"].astype(int)
augmented_df["log_rolled_cases"] = np.log(augmented_df["rolled_cases"] + 1.1)
augmented_df = augmented_df.sort_values(by=["fips","date"])
augmented_df["shifted_log_rolled_cases"] = augmented_df.groupby("fips")["log_rolled_cases"].shift(-7)

/home/zwang937/.local/lib/python3.8/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [3]:
# Check for gaps
gt_columns = ["fips", "county", "state", "days_from_start", "date", "rolled_cases", "log_rolled_cases", "shifted_log_rolled_cases"]
augmented_df_gt = augmented_df[gt_columns]
grouped = augmented_df_gt.groupby('fips')

for fips, group in grouped:
    missing_days = group['days_from_start'].diff().gt(1).sum()
    if missing_days > 0:
        print(f"Gap(s) found in 'days_from_start' for fips {fips}: {missing_days} gap(s)")


### Load and Compile Old Backtests

In [4]:
DATA_FOLDER = "../../data/output/backtest_Imputed_TLGRF_state_forests_windowsize=2"
file_paths = [os.path.join(DATA_FOLDER, file) for file in os.listdir(DATA_FOLDER) if file.endswith(".csv")]

def read_csv(file_path):
    return pd.read_csv(file_path)

with tqdm(total=len(file_paths), desc="Reading CSV files") as pbar:
    # Define the function for reading CSV files
    # Read the CSV files in parallel
    TLGRF_dfs = Parallel(n_jobs=-1)(delayed(read_csv)(file_path) for file_path in file_paths)
    pbar.update(len(file_paths))  # Manually update the progress bar


Reading CSV files:   0%|          | 0/1104 [00:00<?, ?it/s]

In [5]:
#TLGRF_dfs[0]

In [6]:
TLGRF_df = pd.concat(TLGRF_dfs).sort_values(by=["fips", "days_from_start"])
TLGRF_df["datetime"] = pd.to_datetime(TLGRF_df["datetime"])
TLGRF_df = TLGRF_df.rename(columns={"datetime":"date", "log_rolled_cases.x": "log_rolled_cases", "predicted.grf.future.last":"TLGRF_predicted_log_rolled_cases", "t0.hat":"intercept_TLGRF", "tau.hat":"r_TLGRF"})
TLGRF_df = TLGRF_df[["fips","days_from_start", "intercept_TLGRF","r_TLGRF"]]
TLGRF_df = pd.merge(TLGRF_df, augmented_df_gt, on=["fips","days_from_start"], how="right")
TLGRF_df["TLGRF_predicted_log_rolled_cases"] = TLGRF_df["r_TLGRF"]*7 + TLGRF_df["log_rolled_cases"]

#TLGRF_df["shifted_log_rolled_cases"] = TLGRF_df.groupby("fips")["log_rolled_cases"].shift(-7)
TLGRF_df.to_csv("benchmark_Imputed_TLGRF_dataset.csv", index=False)
display(TLGRF_df)

,fips,days_from_start,intercept_TLGRF,r_TLGRF,county,state,date,rolled_cases,log_rolled_cases,shifted_log_rolled_cases,TLGRF_predicted_log_rolled_cases
0,1001,69,NaN,NaN,Autauga,Alabama,2020-03-30,5.142857,1.831438,2.469309,NaN
1,1001,70,NaN,NaN,Autauga,Alabama,2020-03-31,6.000000,1.960095,2.528012,NaN
2,1001,71,NaN,NaN,Autauga,Alabama,2020-04-01,6.857143,2.074070,2.550561,NaN
3,1001,72,NaN,NaN,Autauga,Alabama,2020-04-02,7.428571,2.143422,2.625703,NaN
4,1001,73,NaN,NaN,Autauga,Alabama,2020-04-03,8.285714,2.239189,2.676117,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3390311,99999,1153,1453.435836,-0.031250,New York City,New York,2023-03-19,13972.285714,9.544910,NaN,9.326161
3390312,99999,1154,1395.458269,-0.038533,New York City,New York,2023-03-20,13317.571429,9.496922,NaN,9.227189
3390313,99999,1155,1333.367863,-0.051428,New York City,New York,2023-03-21,12458.714286,9.430264,NaN,9.070270
3390314,99999,1156,1523.737455,-0.025234,New York City,New York,2023-03-22,12154.857143,9.405575,NaN,9.228940


In [7]:
TLGRF_df.head(8)

,fips,days_from_start,intercept_TLGRF,r_TLGRF,county,state,date,rolled_cases,log_rolled_cases,shifted_log_rolled_cases,TLGRF_predicted_log_rolled_cases
0,1001,69,NaN,NaN,Autauga,Alabama,2020-03-30,5.142857,1.831438,2.469309,NaN
1,1001,70,NaN,NaN,Autauga,Alabama,2020-03-31,6.000000,1.960095,2.528012,NaN
2,1001,71,NaN,NaN,Autauga,Alabama,2020-04-01,6.857143,2.074070,2.550561,NaN
3,1001,72,NaN,NaN,Autauga,Alabama,2020-04-02,7.428571,2.143422,2.625703,NaN
4,1001,73,NaN,NaN,Autauga,Alabama,2020-04-03,8.285714,2.239189,2.676117,NaN
5,1001,74,NaN,NaN,Autauga,Alabama,2020-04-04,9.142857,2.326581,2.742682,NaN
6,1001,75,NaN,NaN,Autauga,Alabama,2020-04-05,10.000000,2.406945,2.805090,NaN
7,1001,76,NaN,NaN,Autauga,Alabama,2020-04-06,10.714286,2.469309,2.863832,NaN


In [8]:
TLGRF_df.shape

(3390316, 11)

In [9]:
TLGRF_df["TLGRF_predicted_log_rolled_cases"].isna().sum()

760022

In [10]:
len(TLGRF_df["days_from_start"].unique()) * len(TLGRF_df["fips"].unique())

3620736

In [11]:
grouped = TLGRF_df.groupby('fips')

for fips, group in grouped:
    missing_days = group['days_from_start'].diff().gt(1).sum()
    if missing_days > 0:
        print(f"Gap(s) found in 'days_from_start' for fips {fips}: {missing_days} gap(s)")


In [12]:
TLGRF_df["fips"].isna().sum()

0

In [13]:
TLGRF_df["r_TLGRF"].max(), TLGRF_df["r_TLGRF"].min()

(0.884773077824469, -0.742958653719417)